# 05 Fine-grained controls: row filters and column masks

Job task 5 of 5. Implements the two columns of the role catalog that had no
implementation at all: **Row-Level Security level** and the per-role column
posture for direct identifiers, PHI quasi identifiers and HL7 V_Restricted.

Everything is derived from `rbac/role_catalog.json`:

| Control | Function | Attached to |
|---|---|---|
| RLS level 3 / 2 / 1 | `{catalog}.rbac.rls_service` | every gold table carrying the cohort column (`curr_service`, `service`, `service_name`) |
| RLS admission level | `{catalog}.rbac.rls_hadm_service_filter_<type>` | every gold fact carrying `hadm_id`, except `fact_service`, which the filter reads |
| RLS subject level | `{catalog}.rbac.rls_subject_service_filter_<type>` | `dim_patient`, through the `rbac.subject_service_map` snapshot |
| Direct identifier | `{catalog}.rbac.mask_direct_<type>` | columns tagged `hipaa_identifiability = phi_direct_identifier` |
| PHI quasi identifier | `{catalog}.rbac.mask_quasi_<type>` | columns tagged `hipaa_identifiability = phi_quasi_identifier` |
| HL7 Very Restricted | `{catalog}.rbac.mask_hl7_vr_<type>` | columns tagged `hl7_confidentiality = V_Very_Restricted` |

Masks cover every schema listed in `masking.masked_schemas` — **gold and silver**. Gold is
the research surface and carries the full taxonomy; silver carries the identifier columns
only (see `jobs/01_apply_tags`), so a schema-level grant cannot hand out a raw identifier to
a role whose catalog policy says `masked`. Row filters stay gold-only: only gold carries
the cohort columns and the keys the id-anchored filters resolve through.

Columns listed in `masking.exempt_columns` keep their tag but are never masked: the
classification is still true, the policy chooses readability. That is currently
`dim_patient.gender` and `dim_patient.anchor_age`, so every role can describe a cohort.

How the levels resolve inside `rls_service`:
- member of a level 3 group (`cdr_admins`, `cdr_data_engineers`) -> every row
- a PSYCH row -> only members of that cohort group (`cdr_researcher_psychiatry`)
- anything else -> every role, i.e. the standard cohort. PEDS / NEONATAL are standard
  rows here: this PoC has one restricted cohort, so that the row filter can be shown
  with a single pair of otherwise identical roles

`is_account_group_member()` (not `is_member()`) is used inside the SQL functions because
that is the documented function for row filters and masks. Groups must therefore be
account groups; workspace-local groups fail closed (everyone sees the masked/restricted view).

Re-run after any tag or catalog change. Views inherit masks and row filters from their
base tables, so only base tables are touched.

**Widgets:** `mode` = `dry_run` | `apply` | `rollback`, `catalog_path`.

In [0]:
import json
import os

dbutils.widgets.dropdown("mode", "dry_run", ["dry_run", "apply", "rollback"])
dbutils.widgets.text(
    "catalog_path",
    "/Workspace/Shared/RMIT_CDR_Capstone/04_Governance/rbac/role_catalog.json",
)

MODE = dbutils.widgets.get("mode")
CATALOG_PATH = dbutils.widgets.get("catalog_path")
assert MODE in ("dry_run", "apply", "rollback"), MODE


def load_catalog(path):
    candidates = [
        path,
        "04_Governance/rbac/role_catalog.json",
        os.path.join(os.getcwd(), "04_Governance/rbac/role_catalog.json"),
    ]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            with open(candidate, "r", encoding="utf-8") as handle:
                return json.load(handle), candidate
    raise FileNotFoundError("role_catalog.json not found: " + ", ".join(candidates))


CATALOG_DEF, CATALOG_SOURCE = load_catalog(CATALOG_PATH)
CATALOG = CATALOG_DEF["catalog"]
GOLD = CATALOG_DEF["gold_schema"]
ROLES = CATALOG_DEF["roles"]
MASKING = CATALOG_DEF["masking"]
RLS = CATALOG_DEF["rls"]
MASKED_SCHEMAS = [str(schema).lower() for schema in MASKING.get("masked_schemas", [GOLD])]
SCHEMA_LIST = ", ".join(f"'{schema}'" for schema in MASKED_SCHEMAS)
print(f"catalog={CATALOG} roles={len(ROLES)} masked_schemas={MASKED_SCHEMAS} source={CATALOG_SOURCE}")

catalog=cdr_mimiciv roles=7 masked_schemas=['gold', 'silver'] source=/Workspace/Shared/RMIT_CDR_Capstone/04_Governance/rbac/role_catalog.json


## 1. Safety check

In [0]:

def is_member(group):
    try:
        return bool(spark.sql(f"SELECT is_member('{group}')").first()[0])
    except Exception:
        return bool(spark.sql(f"SELECT is_account_group_member('{group}')").first()[0])


assert is_member("cdr_admins"), "Run this as a member of cdr_admins."
print("membership check passed")

membership check passed


## 2. Discover the gold columns the tag taxonomy protects

In [0]:
TYPE_MAP = {
    "INT": "int", "INTEGER": "int", "BIGINT": "bigint", "LONG": "bigint",
    "SMALLINT": "smallint", "TINYINT": "tinyint",
    "STRING": "string", "VARCHAR": "string", "CHAR": "string",
    "DOUBLE": "double", "FLOAT": "float", "DECIMAL": "decimal(38,18)",
    "DATE": "date", "TIMESTAMP": "timestamp", "BOOLEAN": "boolean", "BINARY": "binary",
}

# (tag_name, tag_value) -> posture key in the catalog's masking.columns
POSTURES = {}
for key, rule in MASKING["columns"].items():
    POSTURES[(rule["tag"], rule["value"])] = key

SHORT = {
    "direct_identifier": "direct",
    "phi_quasi_identifier": "quasi",
    "hl7_vr": "hl7_vr",
}

# `deferred_columns` is a map keyed by schema. A flat list is the original form and only ever
# applied to gold; it is still accepted so that an older catalog cannot silently widen the set.
raw_deferred = MASKING.get("deferred_columns", {})
if isinstance(raw_deferred, dict):
    deferred = {(str(schema).lower(), str(column).lower())
                for schema, columns in raw_deferred.items() for column in columns}
else:
    deferred = {(GOLD.lower(), str(column).lower()) for column in raw_deferred}

exempt = {(
    str(entry.get("schema", GOLD)).lower(),
    str(entry["table"]).lower(),
    str(entry["column"]).lower(),
) for entry in MASKING.get("exempt_columns", [])}

tagged = spark.sql(f"""
    SELECT c.table_schema, c.table_name, c.column_name, c.data_type, ct.tag_name, ct.tag_value
    FROM {CATALOG}.information_schema.columns c
    JOIN {CATALOG}.information_schema.tables t
      ON t.table_schema = c.table_schema AND t.table_name = c.table_name AND t.table_type <> 'VIEW'
    JOIN {CATALOG}.information_schema.column_tags ct
      ON ct.schema_name = c.table_schema AND ct.table_name = c.table_name
     AND ct.column_name = c.column_name
    WHERE c.table_schema IN ({SCHEMA_LIST})
""").collect()

mask_targets = {}   # (schema, table, column) -> {posture, type}
deferred_hits = []
exempt_hits = []
unsupported = []

for row in tagged:
    record = row.asDict()
    posture = POSTURES.get((str(record["tag_name"]), str(record["tag_value"])))
    if posture is None:
        continue
    schema = str(record["table_schema"])
    table = str(record["table_name"])
    column = str(record["column_name"])
    key = (schema, table, column)
    if key in mask_targets:
        continue                      # direct identifier wins over a second tag
    if (schema.lower(), table.lower(), column.lower()) in exempt:
        exempt_hits.append((schema, table, column, posture))   # tagged, deliberately not masked
        continue
    if (schema.lower(), column.lower()) in deferred:
        deferred_hits.append((schema, table, column, posture))
        continue
    column_type = TYPE_MAP.get(str(record["data_type"]).upper().split("(")[0])
    if column_type is None:
        unsupported.append((schema, table, column, str(record["data_type"])))
        continue
    mask_targets[key] = {"posture": posture, "type": column_type}

print(f"{len(mask_targets)} columns to mask, {len(exempt_hits)} exempt by policy, "
      f"{len(deferred_hits)} deferred, {len(unsupported)} unsupported types")

78 columns to mask, 2 exempt by policy, 14 deferred, 0 unsupported types


## 3. Build the mask functions

One function per (posture, type). The function itself resolves the role
posture, so a single attached mask serves every group in the catalog.

In [0]:

def groups_with(posture, value):
    return sorted(role for role, spec in ROLES.items() if spec["policy"].get(posture) == value)


def membership_clause(groups):
    return " OR ".join(f"is_account_group_member('{group}')" for group in groups)


def mask_function_sql(posture, column_type):
    unmasked_groups = groups_with(posture, "unmasked")
    redacted_groups = groups_with(posture, "redacted")
    redaction = MASKING.get("redaction", {}).get(column_type, "NULL")

    branches = []
    if unmasked_groups:
        branches.append(f"WHEN {membership_clause(unmasked_groups)} THEN v")
    if redacted_groups:
        branches.append(f"WHEN {membership_clause(redacted_groups)} THEN {redaction}")

    name = f"{CATALOG}.rbac.mask_{SHORT[posture]}_{column_type.split('(')[0]}"
    comment = (
        f"{posture}: unmasked for {', '.join(unmasked_groups) or 'nobody'}"
        + (f"; redacted for {', '.join(redacted_groups)}" if redacted_groups else "")
        + "; masked (NULL) for every other catalog role"
    )
    sql = (
        f"CREATE OR REPLACE FUNCTION {name}(v {column_type}) RETURNS {column_type}\n"
        f"COMMENT '{comment}'\n"
        "RETURN CASE\n  " + "\n  ".join(branches) + "\n  ELSE NULL\nEND"
    )
    return name, sql


MASK_FUNCTIONS = {}
for target in mask_targets.values():
    key = (target["posture"], target["type"])
    if key not in MASK_FUNCTIONS:
        MASK_FUNCTIONS[key] = mask_function_sql(*key)

print(f"{len(MASK_FUNCTIONS)} mask functions")

7 mask functions


## 4. Build the row filter (RLS levels)

In [0]:
level3 = sorted(role for role, spec in ROLES.items() if spec["rls"]["level"] == 3)
cohorts = RLS["cohorts"]
candidates = [c.lower() for c in RLS["anchor_column_candidates"]]

cohort_branches = []
for cohort_name in sorted(cohorts):
    cohort = cohorts[cohort_name]
    values = ", ".join(f"'{value}'" for value in cohort["values"])
    cohort_branches.append(
        f"WHEN upper(trim(service)) IN ({values}) THEN is_account_group_member('{cohort['group']}')"
    )

RLS_FUNCTION = f"{CATALOG}.rbac.rls_service"
rls_function_sql = (
    f"CREATE OR REPLACE FUNCTION {RLS_FUNCTION}(service STRING) RETURNS BOOLEAN\n"
    "COMMENT 'Row filter for the CDR role catalog: level 3 = all rows, level 2 = standard + own cohort, level 1 = standard rows only'\n"
    "RETURN CASE\n"
    f"  WHEN {membership_clause(level3)} THEN TRUE\n"
    "  WHEN service IS NULL THEN TRUE\n  "
    + "\n  ".join(cohort_branches)
    + "\n  ELSE TRUE\nEND"
)

anchor_rows = spark.sql(f"""
    SELECT c.table_name, c.column_name, t.table_type
    FROM {CATALOG}.information_schema.columns c
    JOIN {CATALOG}.information_schema.tables t
      ON t.table_schema = c.table_schema AND t.table_name = c.table_name
    WHERE c.table_schema = '{GOLD}'
""").collect()

anchor_targets = {}
for row in anchor_rows:
    if str(row["table_type"]).upper() == "VIEW":
        continue
    column = str(row["column_name"]).lower()
    table = str(row["table_name"])
    if column not in candidates:
        continue
    best = candidates.index(column)
    if table not in anchor_targets or best < anchor_targets[table][0]:
        anchor_targets[table] = (best, column)

print(f"{len(anchor_targets)} gold tables carry a cohort column")

gold_tables = {str(row["table_name"]) for row in anchor_rows}
uncovered = sorted(gold_tables - set(anchor_targets))
if uncovered:
    print(f"NOTE: {len(uncovered)} gold tables have no cohort column and cannot be filtered: {uncovered}")

1 gold tables carry a cohort column
NOTE: 27 gold tables have no cohort column and cannot be filtered: ['dim_admission_type', 'dim_careunit', 'dim_date', 'dim_diagnosis', 'dim_icu_item', 'dim_lab_item', 'dim_location', 'dim_medication', 'dim_medication_route', 'dim_microbiology_test', 'dim_order_type', 'dim_patient', 'dim_procedure', 'dim_provider', 'fact_admission', 'fact_diagnosis', 'fact_icu_observation', 'fact_icu_stay', 'fact_lab', 'fact_medication_administration', 'fact_microbiology', 'fact_order', 'fact_order_detail', 'fact_prescription', 'fact_procedure', 'fact_service', 'fact_transfer']


## 5. Assemble and apply

## 4b. Anchor the same cohort rule on keys, not on a service column

`rls_service` needs the service column inside the table it protects, and only `dim_service`
and `fact_service` have one. The other 26 gold tables are therefore unreachable by it, so a
level 1 role can read every admission in `fact_admission`. These two filters close that gap
without inventing a second policy: same cohort, same group, same permissive default.

| Filter | Anchor | Resolves through |
|---|---|---|
| `rls_hadm_service_filter_<type>` | `hadm_id` | `fact_service` -> `dim_service` |
| `rls_subject_service_filter_<type>` | `subject_id` | `rbac.subject_service_map` |

Three constraints shape the code below:
- **A filter may never read the table it protects**, or Unity Catalog recurses. `fact_service`
  and `dim_service` are named in each filter's `reads` list and are refused as targets.
- **One filter per table.** A table already carrying `rls_service` is skipped, not double-filtered.
- **The parameter type must match the anchor column**, and `hadm_id` is INT on nine gold facts
  and BIGINT on three, so one function name per type is generated. This is the same reason the
  mask functions are suffixed.

The subject filter reads a physical snapshot rather than joining `dim_patient`, because a
function that joins the table it filters recurses and a view would inherit that filter anyway.
`subject_service_map` is therefore stale after any Gold rebuild: re-running this notebook
refreshes it. Rebuilding it as `cdr_admins` is what keeps the snapshot complete, since the
permissive branch of every filter is true for that group.

In [0]:
ID_FILTERS = RLS.get("id_anchored_filters", {})

gold_types = {}
for row in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = '{GOLD}'
""").collect():
    gold_types[(str(row["table_name"]), str(row["column_name"]).lower())] = TYPE_MAP.get(
        str(row["data_type"]).upper().split("(")[0], str(row["data_type"]).lower())


def cohort_values(name):
    return ", ".join("'" + str(value) + "'" for value in cohorts[name]["values"])


def cohort_guard(name):
    # Level 3 comes first on purpose: an admin who also sits in the cohort group is still an admin,
    # and a bare `NOT is_account_group_member(<cohort>)` would quietly hide rows from them.
    return (f"({membership_clause(level3)})"
            f" OR NOT is_account_group_member('{cohorts[name]['group']}')")


def map_filter_sql(function, column_type, spec):
    # The body reads ONLY the rbac snapshot. A row filter that joins a gold table which itself
    # carries a filter is rejected by Unity Catalog with UNSUPPORTED_NESTED_ROW_OR_COLUMN_ACCESS_POLICY
    # and takes the whole table offline, so no filter may touch gold at query time.
    return (
        f"CREATE OR REPLACE FUNCTION {CATALOG}.rbac.{function}({spec['param']} {column_type}) RETURNS BOOLEAN\n"
        f"COMMENT '{spec['label']} row filter: {spec['cohort']} rows are visible only to {cohorts[spec['cohort']]['group']} and the level 3 roles; every other caller is permissive. Reads rbac.{spec['map_table']} only.'\n"
        "RETURN\n"
        f"  {cohort_guard(spec['cohort'])}\n"
        "  OR EXISTS (\n"
        "    SELECT 1\n"
        f"    FROM {CATALOG}.rbac.{spec['map_table']} AS m\n"
        f"    WHERE m.{spec['anchor_column']} = {spec['param']}\n"
        f"      AND m.service IN ({cohort_values(spec['cohort'])})\n"
        "  )"
    )


def map_sql(spec):
    source = spec["map_from"]
    keys = " OR ".join(f"ds.{source['dim_key']} = fs.{key}" for key in source["fact_keys"])
    if spec.get("kind") == "hadm":
        body = (
            "SELECT DISTINCT\n"
            f"  fs.{source['output_column']},\n"
            f"  upper(trim(ds.{source['service_column']})) AS service\n"
            f"FROM {CATALOG}.{GOLD}.{source['fact_table']} AS fs\n"
            f"JOIN {CATALOG}.{GOLD}.{source['dim_table']} AS ds ON {keys}\n"
            f"WHERE ds.{source['service_column']} IS NOT NULL"
        )
    else:
        body = (
            "SELECT DISTINCT\n"
            f"  s.{source['output_column']},\n"
            f"  upper(trim(ds.{source['service_column']})) AS service\n"
            f"FROM {CATALOG}.{GOLD}.{source['subject_table']} AS s\n"
            f"JOIN {CATALOG}.{GOLD}.{source['fact_table']} AS fs"
            f" ON fs.{source['fact_key']} = s.{source['subject_join']}\n"
            f"JOIN {CATALOG}.{GOLD}.{source['dim_table']} AS ds ON {keys}\n"
            f"WHERE ds.{source['service_column']} IS NOT NULL"
        )
    return f"CREATE OR REPLACE TABLE {CATALOG}.rbac.{spec['map_table']} AS\n" + body


id_map_statements = []
id_function_statements = []
id_filter_statements = []
id_targets = {}

for key, spec in sorted(ID_FILTERS.items()):
    anchor = str(spec["anchor_column"]).lower()
    reads = {str(name).lower() for name in spec.get("reads", [])}

    if spec.get("map_table") and MODE != "rollback":
        id_map_statements.append((f"map {spec['map_table']}", map_sql(spec)))

    builder = map_filter_sql
    targets, missing, refused = [], [], []
    for table in sorted(spec["tables"]):
        column_type = gold_types.get((table, anchor))
        if column_type is None:
            missing.append(table)
            continue
        if table.lower() in reads:
            refused.append(table)
            continue
        if table in anchor_targets:
            print(f"SKIP {table}: already carries {RLS_FUNCTION}; Unity Catalog allows one filter per table")
            continue
        targets.append((table, column_type))

    if missing:
        print(f"NOTE {key}: declared but not built with {anchor}: {', '.join(missing)}")
    if refused:
        print(f"REFUSED {key}: would read itself and recurse: {', '.join(refused)}")

    functions = {}
    for column_type in sorted({column_type for _, column_type in targets}):
        function = f"{spec['function_prefix']}_{column_type.split('(')[0]}"
        functions[column_type] = function
        if MODE != "rollback":
            id_function_statements.append((f"function {function}", builder(function, column_type, spec)))

    for table, column_type in targets:
        if MODE == "rollback":
            sql = f"ALTER TABLE {CATALOG}.{GOLD}.`{table}` DROP ROW FILTER"
        else:
            sql = (f"ALTER TABLE {CATALOG}.{GOLD}.`{table}` SET ROW FILTER "
                   f"{CATALOG}.rbac.{functions[column_type]} ON (`{spec['anchor_column']}`)")
        id_filter_statements.append((f"row filter {table}", sql))

    id_targets[key] = (targets, functions)

print(f"id-anchored filters: {len(ID_FILTERS)} declared, "
      f"{sum(len(targets) for targets, _ in id_targets.values())} table(s) targeted")

id-anchored filters: 2 declared, 12 table(s) targeted


## 5. Assemble and apply

In [0]:
statements = []
statements.append((f"schema {CATALOG}.rbac", f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.rbac"))

# The subject filter reads this snapshot, so it is built before the functions that reference it.
if MODE == "rollback":
    if id_map_statements:
        print(f"NOTE rollback leaves {len(id_map_statements)} map table(s) in place; drop them by hand")
else:
    for label, sql in id_map_statements:
        statements.append((label, sql))

if MODE != "rollback":
    statements.append(("function rls_service", rls_function_sql))
    for (posture, column_type) in sorted(MASK_FUNCTIONS):
        name, sql = MASK_FUNCTIONS[(posture, column_type)]
        statements.append((f"function {name}", sql))
    for label, sql in id_function_statements:
        statements.append((label, sql))

for (schema, table, column), target in sorted(mask_targets.items()):
    function_name = f"{CATALOG}.rbac.mask_{SHORT[target['posture']]}_{target['type'].split('(')[0]}"
    if MODE == "rollback":
        sql = f"ALTER TABLE {CATALOG}.{schema}.`{table}` ALTER COLUMN `{column}` DROP MASK"
    else:
        sql = f"ALTER TABLE {CATALOG}.{schema}.`{table}` ALTER COLUMN `{column}` SET MASK {function_name}"
    statements.append((f"mask {schema}.{table}.{column}", sql))

for table, (_, column) in sorted(anchor_targets.items()):
    if MODE == "rollback":
        sql = f"ALTER TABLE {CATALOG}.{GOLD}.`{table}` DROP ROW FILTER"
    else:
        sql = f"ALTER TABLE {CATALOG}.{GOLD}.`{table}` SET ROW FILTER {RLS_FUNCTION} ON (`{column}`)"
    statements.append((f"row filter {table}", sql))

for label, sql in id_filter_statements:
    statements.append((label, sql))

print(f"{len(statements)} statements (mode={MODE})")

105 statements (mode=apply)


In [0]:
applied, errors = 0, []

for label, sql in statements:
    if MODE == "dry_run":
        print(sql)
        continue
    try:
        spark.sql(sql)
        applied += 1
    except Exception as exc:
        errors.append((label, str(exc).splitlines()[0][:200]))

print(f"\nmode={MODE} statements={len(statements)} applied={applied} errors={len(errors)}")
for label, message in errors:
    print("ERROR:", label, "->", message)


mode=apply statements=105 applied=105 errors=0


## 6. Evidence

In [0]:
for schema, table, column, posture in sorted(exempt_hits):
    entry = next((e for e in MASKING["exempt_columns"]
                  if str(e.get("schema", GOLD)).lower() == schema.lower()
                  and str(e["table"]).lower() == table.lower()
                  and str(e["column"]).lower() == column.lower()), {})
    print(f"EXEMPT {schema}.{table}.{column} ({posture}): readable for every catalog role "
          f"- {entry.get('reason', 'declared in masking.exempt_columns')}")
for schema, table, column, posture in sorted(deferred_hits):
    print(f"DEFERRED {schema}.{table}.{column} ({posture}): team decision outstanding")
for schema, table, column, data_type in sorted(unsupported):
    print(f"UNSUPPORTED {schema}.{table}.{column} ({data_type})")

if mask_targets:
    display(spark.createDataFrame(
        [
            {"schema": schema, "table": table, "column": column,
             "posture": target["posture"], "type": target["type"]}
            for (schema, table, column), target in sorted(mask_targets.items())
        ]
    ))

if applied:
    for key, (targets, functions) in sorted(id_targets.items()):
        if targets:
            print(f"RLS {key}: {len(targets)} table(s) via "
                  f"{', '.join(sorted(set(functions.values())))}")
    display(spark.sql(f"SELECT * FROM {CATALOG}.information_schema.column_masks WHERE table_schema IN ({SCHEMA_LIST})"))
    display(spark.sql("SELECT * FROM {}.information_schema.row_filters".format(CATALOG)))

assert not errors, f"{len(errors)} statement(s) failed; see output"
dbutils.notebook.exit(
    f"05_fine_grained_controls mode={MODE} masks={len(mask_targets)} "
    f"row_filters={len(anchor_targets)} "
    f"id_filters={sum(len(targets) for targets, _ in id_targets.values())} errors={len(errors)}"
)